# 主動型投資- 資料爬蟲
**以「週」為單位做台股的買賣，賺價差**  
數據(yahoo finance上個股的資訊)：
*  日期(每周最後交易日)
*  股價(每週最後交易日當天收盤價)
*  大戶(三大法人)買賣超張數
*  大戶籌碼比例
*  借券賣出餘額樟張數
*  融資餘額張數
*  券資比
*  融資維持率




## 取得資料
數據來源url (ABBR為台股代碼；若為上櫃/興櫃股票，url中的TW要改為TWO)：
* 法人買賣超：https://tw.finance.yahoo.com/quote/ABBR.TW/institutional-trading (要選則「週」)
* 資券變化：https://tw.finance.yahoo.com/quote/ABBR.TW/margin (要選擇「資券餘額/借券賣出餘額」&「週」)
* 大戶籌碼：https://tw.finance.yahoo.com/quote/ABBR.TW/major-holders
* 股價(週K線)：https://tw.stock.yahoo.com/quote/ABBR.TW/technical-analysis (要選擇「週」；從 K 線圖上方的資訊列讀取 日期/開/高/低/收)

**實際的取得方式 (預設)：JSON 介面版**  
直接呼叫上面這些網頁背後的資料介面 (`https://tw.stock.yahoo.com/_td-stock/api/resource/...`，網頁點「週」時也是呼叫它們)，不開瀏覽器，每檔股票 5 份資料約 2 秒，內容與網頁表格相同：
* 法人買賣超：`StockServices.tradesWithQuoteStats;period=week` (最近 52 週)
* 資券餘額、借券賣出餘額：`StockServices.creditsWithQuoteStats;period=week` (同一個請求，最近 30 週)
* 大戶籌碼：`StockServices.holderStats` (最近一年)
* 股價週K線：`FinanceChartService.ApacLibraCharts;period=w` (最近 120 週)
* 確認不是 Yahoo 的舊快取：同一組介面的日資料 (`period=day` / `period=d`)，週資料的最新一週要和最新交易日同一週

「成交量」在介面版一律是完整數字 (網頁上 100 萬張以上會顯示成 1.019M，瀏覽器版照存)。「漲跌幅(%)」在介面版有正負號 (瀏覽器版沒有，因為網頁用顏色表示漲跌)。注意這欄是**該週最後一個交易日的單日漲跌**，不是整週漲跌；整週漲跌請用股價週K線的收盤價計算 (本週收盤 ÷ 上週收盤 − 1)。

Yahoo 介面失效時，可改用下方的瀏覽器版爬蟲 (`source="browser"`，每檔約 1 分鐘)。

### 匯入Package

In [1]:
import asyncio
import json
import math
import re
import sys
import threading
import time
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path
from statistics import median
from urllib.parse import quote as url_quote

import pandas as pd
import requests
from playwright.async_api import async_playwright

### 爬蟲函式定義：共用函式、瀏覽器版爬蟲 (備案；法人買賣超、資券餘額/借券賣出餘額、大戶籌碼、股價週K線)
瀏覽器版的函式名稱結尾是 `_browser`；平常使用下方的 JSON 介面版 (函式名稱沒有 `_browser`)

In [2]:
# ===== 共用：變數與函式 =====
# (輸入代碼、開網頁、點按鈕、確認週資料、讀表格、跑 Playwright、存檔；下方各資料的爬蟲都會用到)

URL_TEMPLATE = "https://tw.finance.yahoo.com/quote/{symbol}/{page}"
SUFFIXES = (".TW", ".TWO")  # 上市用 .TW；上櫃、興櫃用 .TWO
SELECTED_BG = "rgb(218, 237, 255)"  # 按鈕被選取時的淡藍底色 (#daedff)
PERIOD_BUTTONS = ("日", "週", "月", "季")
STALE_RETRIES = 3  # 週資料是 Yahoo 舊快取時，最多重新載入頁面幾次
STALE_RETRY_WAIT_S = 10  # 每次重新載入前等幾秒
DATA_DIR = "DataCollection"  # 各股票資料夾 (ABBR) 的上層資料夾 (相對於 notebook 所在的資料夾)


def ask_stock_code():
    """請使用者輸入 4 位數字的台股代碼"""
    while True:
        abbr = input("請輸入台股代碼(4位數字)：").strip()
        if re.fullmatch(r"\d{4}", abbr):
            return abbr
        print(f"「{abbr}」不是 4 位數字，請重新輸入。")


def looks_weekly(dates):
    """用表格日期判斷是否為「週」資料：每列落在不同週，且相鄰兩列間隔的中位數約 7 天"""
    if len(dates) < 3:
        return False
    gaps = [(a - b).days for a, b in zip(dates, dates[1:])]
    iso_weeks = [d.isocalendar()[:2] for d in dates]
    return all(g > 0 for g in gaps) and len(set(iso_weeks)) == len(iso_weeks) and 5 <= median(gaps) <= 9


def looks_daily(dates):
    """用表格日期判斷是否為「日」資料：日期由新到舊，相鄰兩列間隔的中位數不超過 3 天 (週末會隔 3 天)"""
    if len(dates) < 3:
        return False
    gaps = [(a - b).days for a, b in zip(dates, dates[1:])]
    return all(g > 0 for g in gaps) and median(gaps) <= 3


class StaleWeekDataError(RuntimeError):
    """Yahoo 偶爾會回傳舊的週資料快取 (例如 11/10 執行，拿到的週資料只算到 10/27)"""


async def _launch_browser(p, headless):
    # 優先使用 Windows 內建的 Edge，失敗再用 Playwright 自帶的 Chromium (需先執行 playwright install chromium)
    try:
        return await p.chromium.launch(channel="msedge", headless=headless)
    except Exception:
        return await p.chromium.launch(headless=headless)


async def _open_section(page, abbr, page_name, heading, timeout_s):
    """依序嘗試 .TW、.TWO 網址，回傳 (實際代號, 標題含 heading 且有表格的區塊)
    page_name 是網址最後一段，例如 institutional-trading、margin、major-holders"""
    tried = []
    for suffix in SUFFIXES:
        symbol = abbr + suffix
        url = URL_TEMPLATE.format(symbol=symbol, page=page_name)
        tried.append(url)
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout_s * 1000)

        # Yahoo 有時會自動導正代號，以頁面標題上的代號為準，例如「環球晶(6488.TWO) 法人買賣」
        m = re.search(rf"\(({abbr}\.TWO?)\)", await page.title())
        if m is None or m.group(1) != symbol:
            continue
        # 沒有這項資料的股票(例如興櫃)會被導回報價頁
        if f"/{page_name}" not in page.url:
            continue

        # 同一頁可能有好幾個標題含 heading 的區塊 (例如大戶籌碼頁)，只取有表格的那個
        section = page.locator("section", has=page.locator("h2", has_text=heading)).filter(
            has=page.locator("li .table-row")
        )
        try:
            await section.wait_for(timeout=timeout_s * 1000)
            return symbol, section
        except Exception:
            continue
    raise RuntimeError(
        f"找不到代碼 {abbr} 的「{heading}」資料(興櫃股票 Yahoo 沒有提供)。已嘗試：\n" + "\n".join(tried)
    )


async def _open_section_fresh(page, abbr, page_name, heading, timeout_s, read_fn):
    """開頁面後執行 read_fn(section) 讀資料，回傳 (實際代號, read_fn 的結果)
    週資料是 Yahoo 的舊快取 (StaleWeekDataError) 就等一下、重新載入頁面再讀"""
    for attempt in range(1, STALE_RETRIES + 1):
        symbol, section = await _open_section(page, abbr, page_name, heading, timeout_s)
        try:
            return symbol, await read_fn(section)
        except StaleWeekDataError as e:
            if attempt == STALE_RETRIES:
                raise StaleWeekDataError(
                    f"{symbol}「{heading}」{e}，重試 {STALE_RETRIES} 次仍是舊資料。"
                    "Yahoo 的週資料可能還沒更新，請過一段時間再執行 (舊的 csv 不會被刪除)。"
                ) from None
            print(f"{symbol}「{heading}」{e}，可能是 Yahoo 的舊快取，{STALE_RETRY_WAIT_S} 秒後重新載入 ({attempt}/{STALE_RETRIES - 1})")
            await asyncio.sleep(STALE_RETRY_WAIT_S)


async def _click_until_selected(page, section, name, group, timeout_s):
    """點擊按鈕 name，直到它的背景變成淡藍色，且同一組 (group) 的其他按鈕都不是淡藍色
    (頁面剛載入時點擊可能沒反應，所以沒變色就重新點)"""
    btn = section.get_by_role("button", name=name, exact=True)
    deadline = time.monotonic() + timeout_s
    while True:
        await btn.click()
        # 按鈕 hover 時也會變淡藍色，先把滑鼠移開以免誤判
        await page.mouse.move(0, 0)
        await asyncio.sleep(1)
        bg = await section.evaluate(
            "sec => Object.fromEntries([...sec.querySelectorAll('button')]"
            ".map(b => [b.innerText.trim(), getComputedStyle(b).backgroundColor]))"
        )
        if bg.get(name) == SELECTED_BG and all(bg.get(o) != SELECTED_BG for o in group if o != name):
            return
        if time.monotonic() > deadline:
            raise RuntimeError(f"等待逾時：「{name}」按鈕沒有變成選取狀態，按鈕底色：{bg}")


def _flatten_row(row):
    """有分組的欄位 (例如「融資」底下的 增減/餘額/使用率％) 讀進來會是 list，
    比照範本 CSV 在前面補一個空白的分組標題欄"""
    flat = []
    for cell in row:
        if isinstance(cell, list):
            flat += [""] + cell
        else:
            flat.append(cell)
    return flat


async def _wait_rows(section, n_cols, timeout_s, check, label):
    """讀取表格直到日期通過 check (looks_weekly / looks_daily)，回傳 (rows, dates)
    (表格可能延遲更新，所以輪詢到符合為止；只保留欄位數為 n_cols 的列)"""
    deadline = time.monotonic() + timeout_s
    while True:
        rows = await section.evaluate(
            "sec => [...sec.querySelectorAll('li .table-row')].map(r => [...r.children].map(c =>"
            " c.querySelector('li') ? [...c.querySelectorAll('li')].map(l => l.innerText.trim())"
            " : c.innerText.trim()))"
        )
        rows = [_flatten_row(r) for r in rows]
        rows = [r for r in rows if len(r) == n_cols]
        dates = [datetime.strptime(r[0], "%Y/%m/%d").date() for r in rows]
        if check(dates):
            return rows, dates
        if time.monotonic() > deadline:
            raise RuntimeError(f"等待逾時：表格日期看起來不是「{label}」資料，請重新執行。")
        await asyncio.sleep(0.5)


async def _wait_weekly_rows(section, n_cols, timeout_s):
    """讀取表格直到日期呈現「每週一筆」，回傳 (rows, dates)"""
    return await _wait_rows(section, n_cols, timeout_s, looks_weekly, "週")


def check_week_fresh(week_dates, latest_day):
    """週資料最新一列要和「日」資料的最新交易日落在同一週，否則視為 Yahoo 回傳的舊快取"""
    if week_dates[0].isocalendar()[:2] != latest_day.isocalendar()[:2]:
        raise StaleWeekDataError(f"週資料最新一列是 {week_dates[0]}，但日資料最新交易日是 {latest_day}")


async def _read_week_checked(page, section, n_cols, timeout_s):
    """先點「日」記下最新交易日，再點「週」讀週資料並確認不是舊快取，回傳 (rows, dates)"""
    await _click_until_selected(page, section, "日", PERIOD_BUTTONS, timeout_s)
    _, daily_dates = await _wait_rows(section, n_cols, timeout_s, looks_daily, "日")
    # 確認方法 1：「週」按鈕背景變成淡藍色，日/月/季按鈕不是淡藍色
    await _click_until_selected(page, section, "週", PERIOD_BUTTONS, timeout_s)
    # 確認方法 2：表格日期呈現「每週一筆」
    rows, dates = await _wait_weekly_rows(section, n_cols, timeout_s)
    # 確認方法 3：最新一週包含最新交易日 (Yahoo 偶爾會回傳好幾週、甚至好幾個月前的週資料快取)
    check_week_fresh(dates, daily_dates[0])
    return rows, dates


def _to_week_df(rows, dates, columns, symbol):
    df = pd.DataFrame(rows, columns=columns)
    # 第一欄是日期，格式比照範本 (2026/10/8，不補 0)
    df[columns[0]] = [f"{d.year}/{d.month}/{d.day}" for d in dates]
    df.attrs["symbol"] = symbol  # 記下實際使用的代號 (例如 6488.TWO)，不會寫進 CSV
    return df


def run_in_own_loop(coro_fn, *args, **kwargs):
    """Jupyter 本身已有 event loop (Windows 上還是不支援子行程的 Selector loop)，
    所以另開一個執行緒、用自己的 event loop 跑 Playwright。"""
    result = {}

    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            result["value"] = loop.run_until_complete(coro_fn(*args, **kwargs))
        except BaseException as e:
            result["error"] = e
        finally:
            loop.close()

    t = threading.Thread(target=worker)
    t.start()
    t.join()
    if "error" in result:
        raise result["error"]
    return result["value"]


def clear_week_csv(abbr, file_tags, base_dir=DATA_DIR):
    """刪除 DataCollection/ABBR 資料夾內舊的 ABBR_{file_tag}_YYYYMMDD.csv (每個 file_tag 不論日期都刪)，回傳刪掉的檔案"""
    folder = Path(base_dir) / abbr
    removed = []
    for file_tag in file_tags:
        for old in folder.glob(f"{abbr}_{file_tag}_*.csv"):
            old.unlink()
            removed.append(old)
    return removed


def save_week_csv(df, abbr, file_tag, base_dir=DATA_DIR):
    """存成 DataCollection/ABBR/ABBR_{file_tag}_YYYYMMDD.csv (UTF-8 BOM、CRLF 換行，比照範本 CSV)"""
    folder = Path(base_dir) / abbr
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{abbr}_{file_tag}_{date.today():%Y%m%d}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig", lineterminator="\r\n")
    return path


# ===== 法人買賣超（週）爬蟲 (瀏覽器版)：函式定義 =====
# 網址 .../institutional-trading，先讀「日」的最新交易日，再點「週」後讀「法人逐日買賣超」表格

INSTITUTIONAL_COLUMNS = ["日期", "外資(張)", "投信(張)", "自營商(張)", "合計(張)", "外資籌碼", "漲跌幅(%)", "成交量"]


async def _scrape_institutional_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            page = await browser.new_page(locale="zh-TW")

            async def read(section):
                return await _read_week_checked(page, section, len(INSTITUTIONAL_COLUMNS), timeout_s)

            symbol, (rows, dates) = await _open_section_fresh(
                page, abbr, "institutional-trading", "法人逐日買賣超", timeout_s, read
            )
        finally:
            await browser.close()
    return _to_week_df(rows, dates, INSTITUTIONAL_COLUMNS, symbol)


def fetch_institutional_trading_week_browser(abbr, headless=True, timeout_s=30):
    """(瀏覽器版) 取得「法人逐日買賣超」→「週」的表格資料，回傳 DataFrame"""
    return run_in_own_loop(_scrape_institutional_week, abbr, headless=headless, timeout_s=timeout_s)


# ===== 資券餘額 / 借券賣出餘額（週）爬蟲 (瀏覽器版)：函式定義 =====
# 網址 .../margin，在「資券餘額逐日增減」依序切到兩個分頁，各自先讀「日」的最新交易日、再點「週」後讀表格

# 欄位比照 Margin_Week_11cols.csv、Securities_Lending_and_Borrowing_Week_8cols.csv
# 「融資」「融券」「借券賣出」是網頁上的分組標題，在 CSV 中是空白欄
MARGIN_COLUMNS = ["日期", "融資", "增減", "餘額", "使用率％", "融券", "增減", "餘額", "使用率％", "券資比", "資券互抵"]
LENDING_COLUMNS = ["日期", "融券", "增減", "餘額", "限額", "借券賣出", "增減", "餘額"]
MARGIN_TABS = {"資券餘額": MARGIN_COLUMNS, "借券賣出餘額": LENDING_COLUMNS}  # 分頁名稱: 欄位


async def _scrape_margin_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            page = await browser.new_page(locale="zh-TW")

            async def read(section):
                results = []
                for tab, columns in MARGIN_TABS.items():
                    # 分頁按鈕要變成淡藍色
                    await _click_until_selected(page, section, tab, MARGIN_TABS, timeout_s)
                    # 兩個分頁的欄位數不同 (11 / 8)，用欄位數篩選就不會讀到切換前的舊表格
                    results.append(await _read_week_checked(page, section, len(columns), timeout_s))
                return results

            symbol, results = await _open_section_fresh(page, abbr, "margin", "資券餘額逐日增減", timeout_s, read)
        finally:
            await browser.close()
    return tuple(
        _to_week_df(rows, dates, columns, symbol) for (rows, dates), columns in zip(results, MARGIN_TABS.values())
    )


def fetch_margin_week_browser(abbr, headless=True, timeout_s=30):
    """(瀏覽器版) 取得「資券餘額逐日增減」→「週」的兩個分頁，回傳 (資券餘額 DataFrame, 借券賣出餘額 DataFrame)"""
    return run_in_own_loop(_scrape_margin_week, abbr, headless=headless, timeout_s=timeout_s)


# ===== 大戶籌碼（週）爬蟲 (瀏覽器版)：函式定義 =====
# 網址 .../major-holders，表格本來就是每週一筆，不用點按鈕，只用表格日期確認

MAJOR_HOLDERS_COLUMNS = ["年度/日期", "外資籌碼", "大戶籌碼", "董監持股", "股價"]  # 比照 Major_Holders_Week_5cols.csv


async def _scrape_major_holders_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            page = await browser.new_page(locale="zh-TW")
            # 區塊標題是「台積電(2330) 大戶籌碼」這種格式，用「大戶籌碼」比對
            symbol, section = await _open_section(page, abbr, "major-holders", "大戶籌碼", timeout_s)
            rows, dates = await _wait_weekly_rows(section, len(MAJOR_HOLDERS_COLUMNS), timeout_s)
        finally:
            await browser.close()
    return _to_week_df(rows, dates, MAJOR_HOLDERS_COLUMNS, symbol)


def fetch_major_holders_week_browser(abbr, headless=True, timeout_s=30):
    """(瀏覽器版) 取得「大戶籌碼」的週表格資料，回傳 DataFrame"""
    return run_in_own_loop(_scrape_major_holders_week, abbr, headless=headless, timeout_s=timeout_s)

In [4]:
# ===== 股價（週K線）爬蟲 (瀏覽器版)：函式定義 =====
# 網址 https://tw.stock.yahoo.com/quote/ABBR.TW(O)/technical-analysis，先讀「日」的最新交易日，再點「週」後，
# 用滑鼠逐一移到每一根 K 棒上，讀取 K 線圖上方資訊列顯示的 日期/開/高/低/收 (K 線圖約顯示最近 120 週)

PRICE_URL_TEMPLATE = "https://tw.stock.yahoo.com/quote/{symbol}/technical-analysis"
PRICE_PERIOD_BUTTONS = ("5分", "15分", "30分", "60分", "日", "週", "月")
PRICE_COLUMNS = ["日期", "開", "高", "低", "收"]
PRICE_FILE_TAG = "Stock_Price_Week"
DAILY_CANDLES = 5  # 讀「日」K 線時只 hover 最右邊幾根，用來取得最新交易日

CANDLE_SELECTOR = "g.highcharts-candlestick-series .highcharts-point"
OHLC_BAR_SELECTOR = ".ohlcList-tooltip"  # K 線圖上方的資訊列：日期 開 高 低 收 量(張) 漲跌

# 第 i 根 K 棒的中心 x、繪圖區中央的 y、繪圖區左右邊界 (每次重新計算，避免廣告載入造成版面位移)
_CANDLE_POS_JS = """i => {
  const plot = document.querySelector('.highcharts-plot-background').getBoundingClientRect();
  const r = document.querySelectorAll('g.highcharts-candlestick-series .highcharts-point')[i].getBoundingClientRect();
  return [r.x + r.width / 2, plot.y + plot.height / 2, plot.x, plot.x + plot.width];
}"""

# 資訊列的每一格是「標籤 + 數值」(例如 開 58.4)，只有日期那格沒有標籤
_READ_OHLC_JS = """bar => [...bar.children].map(d => d.children.length >= 2
  ? [d.children[0].innerText.trim(), d.children[1].innerText.trim()]
  : ['日期', d.innerText.trim()])"""

# 圖表本身的 K 棒數值，只用來核對滑鼠讀到的數值
_CHART_POINTS_JS = """() => {
  const c = window.Highcharts && Highcharts.charts.filter(Boolean)[0];
  return c ? c.series[0].points.map(p => [p.x, p.open, p.high, p.low, p.close]) : null;
}"""


async def _open_technical_page(page, abbr, timeout_s):
    """依序嘗試 .TW、.TWO 網址，回傳實際代號 (例如 5498.TWO)"""
    tried = []
    for suffix in SUFFIXES:
        symbol = abbr + suffix
        url = PRICE_URL_TEMPLATE.format(symbol=symbol)
        tried.append(url)
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout_s * 1000)
        # 上櫃股票用 .TW 網址也會顯示頁面，但標題上的代號是 .TWO，例如「凱崴(5498.TWO) 技術分析」
        m = re.search(rf"\(({abbr}\.TWO?)\)", await page.title())
        if m is None or m.group(1) != symbol or "/technical-analysis" not in page.url:
            continue
        try:
            await page.locator(CANDLE_SELECTOR).first.wait_for(timeout=timeout_s * 1000)
            return symbol
        except Exception:
            continue
    raise RuntimeError(f"找不到代碼 {abbr} 的技術分析 K 線圖。已嘗試：\n" + "\n".join(tried))


async def _read_ohlc_bar(page):
    return dict(await page.locator(OHLC_BAR_SELECTOR).evaluate(_READ_OHLC_JS))


async def _hover_candles(page, timeout_s, last_n=None):
    """滑鼠由左到右移到每一根 K 棒上 (有給 last_n 時只移到最右邊 last_n 根)，讀取資訊列，
    回傳 [{日期, 開, 高, 低, 收}, ...]"""
    await page.locator(".highcharts-plot-background").first.scroll_into_view_if_needed()
    n_candles = await page.locator(CANDLE_SELECTOR).count()
    start = 0 if last_n is None else max(0, n_candles - last_n)
    prev_date = (await _read_ohlc_bar(page))["日期"]  # 還沒 hover 時顯示最新一根
    records = []
    for i in range(start, n_candles):
        x, y, left, right = await page.evaluate(_CANDLE_POS_JS, i)
        if not left <= x <= right:  # 只露出一半、中心在繪圖區外的 K 棒無法 hover
            continue
        # 相鄰兩根 K 棒的日期一定不同：資訊列的日期變了，才代表已讀到這一根
        deadline = time.monotonic() + timeout_s
        wiggle = 0
        while True:
            await page.mouse.move(x, y + wiggle)
            bar = await _read_ohlc_bar(page)
            if bar["日期"] != prev_date:
                break
            if time.monotonic() > deadline:
                raise RuntimeError(f"等待逾時：滑鼠移到第 {i + 1} 根 K 棒後，資訊列沒有更新")
            wiggle = 1 - wiggle  # 在原地上下移動 1px，觸發 mousemove
            await asyncio.sleep(0.1)
        prev_date = bar["日期"]
        records.append({col: bar[col] for col in PRICE_COLUMNS})
    return records


async def _read_candles(page, period, check, timeout_s, last_n=None):
    """點週期按鈕 period，hover K 棒並讀取數值，回傳 (records, 日期由新到舊)"""
    # 確認方法 1：週期按鈕背景變成淡藍色，其他週期按鈕不是淡藍色
    await _click_until_selected(page, page.locator("body"), period, PRICE_PERIOD_BUTTONS, timeout_s)
    # 確認方法 2：K 棒的日期通過 check (looks_weekly / looks_daily)；圖表可能還在重繪，所以輪詢到符合為止
    deadline = time.monotonic() + timeout_s
    while True:
        await asyncio.sleep(1.5)
        records = await _hover_candles(page, timeout_s, last_n)
        dates = sorted({datetime.strptime(r["日期"], "%Y/%m/%d").date() for r in records}, reverse=True)
        if len(dates) == len(records) and check(dates):
            return records, dates
        if time.monotonic() > deadline:
            raise RuntimeError(f"等待逾時：K 線圖的日期看起來不是「{period}」資料，請重新執行。")


async def _scrape_stock_price_week(abbr, headless=True, timeout_s=30):
    async with async_playwright() as p:
        browser = await _launch_browser(p, headless)
        try:
            # 視窗要夠高，整個 K 線圖才能捲到畫面內讓滑鼠 hover
            page = await browser.new_page(locale="zh-TW", viewport={"width": 1400, "height": 1200})
            # 週資料是 Yahoo 的舊快取 (StaleWeekDataError) 就等一下、重新載入頁面再讀
            for attempt in range(1, STALE_RETRIES + 1):
                symbol = await _open_technical_page(page, abbr, timeout_s)
                # 先點「日」記下最新交易日，再點「週」讀每一根 K 棒
                _, daily_dates = await _read_candles(page, "日", looks_daily, timeout_s, last_n=DAILY_CANDLES)
                records, dates = await _read_candles(page, "週", looks_weekly, timeout_s)
                try:
                    # 確認方法 3：最新一週包含最新交易日 (Yahoo 偶爾會回傳好幾週前的週資料快取)
                    check_week_fresh(dates, daily_dates[0])
                    break
                except StaleWeekDataError as e:
                    if attempt == STALE_RETRIES:
                        raise StaleWeekDataError(
                            f"{symbol} 週K線{e}，重試 {STALE_RETRIES} 次仍是舊資料。"
                            "Yahoo 的週資料可能還沒更新，請過一段時間再執行 (舊的 csv 不會被刪除)。"
                        ) from None
                    print(f"{symbol} 週K線{e}，可能是 Yahoo 的舊快取，{STALE_RETRY_WAIT_S} 秒後重新載入 ({attempt}/{STALE_RETRIES - 1})")
                    await asyncio.sleep(STALE_RETRY_WAIT_S)
            chart_points = await page.evaluate(_CHART_POINTS_JS)
        finally:
            await browser.close()
    return symbol, records, chart_points


def _to_number(s):
    return float(s.replace(",", ""))


def _check_against_chart(df, chart_points):
    """用圖表本身的 K 棒數值核對滑鼠讀到的開高低收，不一致就報錯"""
    if chart_points is None:
        print("(找不到頁面上的 Highcharts 物件，略過數值核對)")
        return
    taipei_tz = timezone(timedelta(hours=8))  # K 棒的時間戳記是台北時間當天 0 點
    chart = {}
    for x, *ohlc in chart_points:
        d = datetime.fromtimestamp(x / 1000, taipei_tz).date()
        chart[f"{d.year}/{d.month}/{d.day}"] = ohlc
    mismatched = [
        row["日期"] for _, row in df.iterrows()
        if row["日期"] not in chart
        or any(abs(_to_number(row[c]) - v) > 1e-6 for c, v in zip(["開", "高", "低", "收"], chart[row["日期"]]))
    ]
    if mismatched:
        raise RuntimeError(f"滑鼠讀到的數值與 K 線圖資料不一致，日期：{mismatched}")
    print(f"✔ {len(df)} 週的開高低收都與 K 線圖資料一致")


def fetch_stock_price_week_browser(abbr, headless=True, timeout_s=30):
    """(瀏覽器版) 取得 K 線圖「週」的 日期/開/高/低/收，回傳 DataFrame (日期新到舊，格式比照其他週資料)"""
    symbol, records, chart_points = run_in_own_loop(
        _scrape_stock_price_week, abbr, headless=headless, timeout_s=timeout_s
    )
    df = pd.DataFrame(records, columns=PRICE_COLUMNS)
    for col in PRICE_COLUMNS[1:]:
        df[col] = df[col].str.replace(",", "")
    df = df.iloc[::-1].reset_index(drop=True)
    dates = [datetime.strptime(s, "%Y/%m/%d").date() for s in df["日期"]]
    df = _to_week_df(df.values.tolist(), dates, PRICE_COLUMNS, symbol)  # 日期改成 2026/10/8 (不補 0)
    _check_against_chart(df, chart_points)
    return df

### 爬蟲函式定義：JSON 介面版 (預設)
直接呼叫 Yahoo 網頁背後的資料介面，不開瀏覽器。函式名稱、參數、回傳的 DataFrame 都與瀏覽器版相同(只有「漲跌幅(%)」多了正負號)，所以 AllStockPreparation、DataProcess 不用修改。`headless=False` (想看瀏覽器操作過程) 時會自動改用瀏覽器版。

In [5]:
# ===== JSON 介面版爬蟲 (預設)：函式定義 =====
# 直接呼叫 Yahoo 股市網頁背後的資料介面 (網頁點「週」按鈕時也是向這些介面要資料)，不開瀏覽器，每檔股票約 2 秒。
# 回傳的 DataFrame 與瀏覽器版相同 (欄位、列數、數字格式都一樣)，只有「漲跌幅(%)」多了正負號。
# 函式名稱、參數都和以前一樣，AllStockPreparation、DataProcess 不用改就會使用這一版；
# headless=False (想看瀏覽器操作過程) 時會自動改用瀏覽器版。
# 注意：本格等號右邊不能呼叫函式 (例如 threading.Lock())，因為其他 notebook 載入本檔時會略過這種設定

API_BASE_URL = "https://tw.stock.yahoo.com/_td-stock/api/resource/"
API_QUERY = {  # 網頁呼叫介面時帶的參數
    "bkt": "", "device": "desktop", "ecma": "modern", "intl": "tw", "lang": "zh-Hant-TW",
    "partner": "none", "region": "TW", "site": "finance", "tz": "Asia/Taipei", "returnMeta": "true",
}
API_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/130.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Accept-Language": "zh-TW,zh;q=0.9",
}
API_MIN_INTERVAL_S = 0.1  # 兩次請求之間至少間隔幾秒 (所有執行緒合計，也就是每秒最多 10 次)
API_RETRIES = 3  # 連線錯誤、429 (請求太頻繁)、5xx 時最多重試幾次
API_RETRY_WAIT_S = 3  # 第一次重試前等幾秒，之後每次加倍
API_DAILY_ROWS = 5  # 讀日資料確認最新交易日時，只取最近幾天
# 週資料是舊快取時，重讀前依序等幾秒 (Yahoo 的資券週資料在第一次被問到時，常回傳上週的快取並在背景更新，
# 通常 1～3 秒後再讀就是新的；全部等完仍是舊的才報錯)
API_STALE_WAITS_S = (2, 4, 8)

# 每份週資料保留的列數，與網頁表格顯示的相同
INSTITUTIONAL_ROWS = 52  # 法人買賣超：最近 52 週
MARGIN_ROWS = 30  # 資券餘額、借券賣出餘額：最近 30 週
MAJOR_HOLDERS_DAYS = 365  # 大戶籌碼：最近一年 (該週的週一在今天往前 365 天內)
PRICE_ROWS = 120  # 股價週K線：最近 120 週
MIN_WEEK_ROWS = 3  # 週資料少於幾週就當作沒有資料 (例如剛上市)，與瀏覽器版相同

SYMBOL_HINTS = {}  # {代號: "2330.TW" 或 "5498.TWO"}；第一次用到時從最新的 All_Taiwan_Stock_YYYYMMDD.csv 載入
_API_STATE = {}  # 連線、節流等共用狀態 (第一次呼叫介面時建立)


class YahooNoDataError(RuntimeError):
    """Yahoo 沒有這份資料 (查無代號、不能融資融券、剛上市不足 3 週…)，重新抓取也不會有"""


# --- 連線、節流、重試 ---
def _api_state():
    if "lock" not in _API_STATE:  # 先建其他項目，最後才建 lock (其他執行緒看到 lock 時，其餘項目一定已建好)
        _API_STATE.setdefault("next_time", 0.0)
        _API_STATE.setdefault("local", threading.local())
        _API_STATE.setdefault("symbol_lock", threading.Lock())
        _API_STATE.setdefault("lock", threading.Lock())
    return _API_STATE


def _api_session():
    """每個執行緒各用一個 requests.Session (可以重複使用連線)"""
    local = _api_state()["local"]
    session = getattr(local, "session", None)
    if session is None:
        session = local.session = requests.Session()
        session.headers.update(API_HEADERS)
    return session


def _api_throttle():
    """讓所有執行緒合計的請求間隔至少 API_MIN_INTERVAL_S 秒"""
    state = _api_state()
    with state["lock"]:
        now = time.monotonic()
        start = max(now, state["next_time"])
        state["next_time"] = start + API_MIN_INTERVAL_S
    if start > now:
        time.sleep(start - now)


def _api_get(resource, what, timeout_s=30):
    """呼叫一個資料介面，回傳 JSON。404 → YahooNoDataError；連線錯誤、429、5xx → 等一下再試"""
    url = API_BASE_URL + resource
    last_error = None
    for attempt in range(API_RETRIES + 1):
        if attempt:
            time.sleep(API_RETRY_WAIT_S * 2 ** (attempt - 1))
        _api_throttle()
        try:
            resp = _api_session().get(url, params=API_QUERY, timeout=timeout_s)
        except requests.RequestException as e:
            last_error = f"{type(e).__name__}: {e}"
            continue
        if resp.status_code == 200:
            try:
                return resp.json()
            except ValueError as e:
                raise RuntimeError(f"Yahoo 介面回傳的 {what} 不是 JSON ({e})") from None
        if resp.status_code == 404:
            raise YahooNoDataError(f"Yahoo 沒有 {what} 的資料 (HTTP 404，可能已下市或代號錯誤)")
        last_error = f"HTTP {resp.status_code} {resp.text[:100]}"
        if resp.status_code != 429 and resp.status_code < 500:
            break  # 其他 4xx 重試也沒用 (例如介面改版)
    raise RuntimeError(f"Yahoo 介面讀取 {what} 失敗：{last_error}")


def _api_format_error(symbol, label, e):
    return RuntimeError(
        f"Yahoo 介面回傳的 {symbol}「{label}」格式和預期不同 ({type(e).__name__}: {e})。"
        '若一直發生，可能是 Yahoo 改版，可暫時改用瀏覽器版 (source="browser")'
    )


def stock_list_symbols(stock_list):
    """股票清單 (All_Taiwan_Stock_YYYYMMDD.csv 的 DataFrame) → {代號: "代號.TW" (上市、創新板) 或 "代號.TWO" (上櫃)}"""
    return {
        code: code + (".TWO" if market == "上櫃" else ".TW")
        for code, market in zip(stock_list["代號"], stock_list["市場別"])
    }


def _api_symbol(abbr):
    """介面用的代號。介面不分 .TW / .TWO 都查得到 (只影響顯示的代號)，有股票清單時照清單上的市場別"""
    if abbr not in SYMBOL_HINTS:
        state = _api_state()
        with state["symbol_lock"]:  # 多個執行緒同時用到時，只讀一次清單，其他執行緒等它讀完
            if not state.get("stock_list_loaded"):
                files = sorted(
                    f for f in Path.cwd().glob("All_Taiwan_Stock_[0-9]*.csv")
                    if re.fullmatch(r"All_Taiwan_Stock_\d{8}\.csv", f.name)
                )
                if files:
                    try:
                        stock_list = pd.read_csv(files[-1], encoding="utf-8-sig", dtype={"代號": str})
                        for code, symbol in stock_list_symbols(stock_list).items():
                            SYMBOL_HINTS.setdefault(code, symbol)
                    except Exception:
                        pass  # 讀不到清單也沒關係，預設用 .TW
                state["stock_list_loaded"] = True
    return SYMBOL_HINTS.get(abbr, abbr + SUFFIXES[0])


# --- 數字格式 (與網頁表格顯示的文字相同) ---
def _js_round(x, digits=2):
    """與 Yahoo 網頁 (numeral.js) 相同的四捨五入：先轉成最短的十進位字串再進位，
    例如使用率 0.01995 → 1.995% → 2.00% (直接用 Python 格式化會得到 1.99%)"""
    shifted = float(Decimal(repr(float(x))).scaleb(digits))
    return math.floor(shifted + 0.5) / 10**digits


def _is_blank(v):
    return v is None or (isinstance(v, str) and v.strip() in ("", "-"))


def _fmt_int(v):
    """1234 → "1,234"、-1234 → "-1,234"；沒有資料 → "-" """
    return "-" if _is_blank(v) else f"{int(round(float(v))):,}"


def _fmt_percent(v):
    """百分比數值 "67.58" → "67.58%" """
    return "-" if _is_blank(v) else f"{_js_round(float(v)):.2f}%"


def _fmt_ratio(v):
    """比例 0.0047 → "0.47%"；負數保留正負號 (-0.0476 → "-4.76%")"""
    return "-" if _is_blank(v) else f"{_js_round(float(v) * 100):.2f}%"


def _fmt_price_text(v):
    """大戶籌碼的股價 (文字) "2000" → "2,000"、"39.80" → "39.80" (小數位數照網頁原樣)"""
    if _is_blank(v):
        return "-"
    whole, _, frac = str(v).partition(".")
    return f"{int(whole):,}" + (f".{frac}" if frac else "")


def _fmt_price_number(v):
    """K 線的開高低收 46.650001 → "46.65"、2145.0 → "2145" (與瀏覽器版相同，不加千分位)"""
    return f"{round(float(v), 2):.2f}".rstrip("0").rstrip(".")


def _fmt_change(row, plus, *minus):
    """增減 = row[plus] - row[minus...]，例如融資增減 = 融資買進 - 融資賣出 - 現金償還；全部沒有資料 → "-" """
    keys = (plus,) + minus
    if all(_is_blank(row.get(k)) for k in keys):
        return "-"
    values = {k: 0.0 if _is_blank(row.get(k)) else float(row.get(k)) for k in keys}
    return _fmt_int(values[plus] - sum(values[k] for k in minus))


def _api_date(text):
    """ "2026/11/26" 或 "2026-11-23T00:00:00+08:00" → date"""
    text = str(text)
    if "/" in text:
        return datetime.strptime(text[:10], "%Y/%m/%d").date()
    return datetime.fromisoformat(text).date()


# --- 檢查週資料 ---
def _api_check_weeks(dates, symbol, label):
    """週資料至少 MIN_WEEK_ROWS 週，而且日期由新到舊、每週一筆"""
    if len(dates) < MIN_WEEK_ROWS:
        raise YahooNoDataError(
            f"Yahoo 的 {symbol}「{label}」只有 {len(dates)} 週資料 (少於 {MIN_WEEK_ROWS} 週，例如剛上市)"
        )
    weeks = [d.isocalendar()[:2] for d in dates]
    if any(a <= b for a, b in zip(dates, dates[1:])) or len(set(weeks)) != len(weeks):
        raise RuntimeError(f"{symbol}「{label}」的日期不是每週一筆：{[str(d) for d in dates[:6]]}…")


def _api_read_fresh(symbol, label, read_week, read_latest_day):
    """read_week() → (資料, 由新到舊的日期)；read_latest_day() → 日資料的最新交易日。
    週資料的最新一週不能比日資料的最新交易日舊，否則視為 Yahoo 的舊快取，依 API_STALE_WAITS_S 等一下再讀。
    週資料的最新一週就是本週時不可能是舊資料，不必再讀日資料"""
    this_week = date.today().isocalendar()[:2]
    waits = list(API_STALE_WAITS_S)
    latest_day = None
    while True:
        data, dates = read_week()
        week = dates[0].isocalendar()[:2]
        if week == this_week:
            return data, dates
        if latest_day is None:
            latest_day = read_latest_day()
        if latest_day is None or week >= latest_day.isocalendar()[:2]:
            return data, dates
        if not waits:
            raise StaleWeekDataError(
                f"{symbol}「{label}」週資料最新一列是 {dates[0]}，但日資料最新交易日是 {latest_day}，"
                f"重讀 {len(API_STALE_WAITS_S)} 次仍是舊資料。Yahoo 的週資料可能還沒更新，請過一段時間再執行 (舊的 csv 不會被刪除)。"
            )
        time.sleep(waits.pop(0))


# --- 各份資料 ---
def _api_institutional(symbol, timeout_s):
    label = "法人逐日買賣超"

    def read_week():
        data = _api_get(
            f"StockServices.tradesWithQuoteStats;limit=60;period=week;symbol={symbol}", f"{symbol} 法人買賣超(週)", timeout_s
        )
        try:
            items = sorted(data["data"]["list"], key=lambda r: _api_date(r["formattedDate"]), reverse=True)
            items = items[:INSTITUTIONAL_ROWS]
            dates = [_api_date(r["formattedDate"]) for r in items]  # 每週最後一個交易日
            rows = [
                [None, _fmt_int(r.get("foreignDiffVolK")), _fmt_int(r.get("investmentTrustDiffVolK")),
                 _fmt_int(r.get("dealerDiffVolK")), _fmt_int(r.get("totalDiffVolK")),
                 _fmt_percent(r.get("foreignHoldPercent")), _fmt_ratio(r.get("changePercent")),
                 _fmt_int(r.get("volumeK"))]
                for r in items
            ]
        except (AttributeError, KeyError, TypeError, ValueError, IndexError) as e:
            raise _api_format_error(symbol, label, e) from None
        _api_check_weeks(dates, symbol, label)
        return rows, dates

    def read_latest_day():
        data = _api_get(
            f"StockServices.tradesWithQuoteStats;limit={API_DAILY_ROWS};period=day;symbol={symbol}",
            f"{symbol} 法人買賣超(日)", timeout_s,
        )
        try:
            return max((_api_date(r["formattedDate"]) for r in data["data"]["list"]), default=None)
        except (AttributeError, KeyError, TypeError, ValueError) as e:
            raise _api_format_error(symbol, label, e) from None

    rows, dates = _api_read_fresh(symbol, label, read_week, read_latest_day)
    return _to_week_df(rows, dates, INSTITUTIONAL_COLUMNS, symbol)


def _api_margin(symbol, timeout_s):
    label = "資券餘額逐日增減"

    def credits(period, limit):
        data = _api_get(
            f"StockServices.creditsWithQuoteStats;dataKey=marginBalance;limit={limit};period={period};symbol={symbol};tabKey={period}",
            f"{symbol} 資券餘額({'週' if period == 'week' else '日'})", timeout_s,
        )
        try:
            return data["data"]["data"]["result"]["credits"] or []
        except (AttributeError, KeyError, TypeError) as e:
            raise _api_format_error(symbol, label, e) from None

    def read_week():
        items = credits("week", MARGIN_ROWS)
        if not items:
            raise YahooNoDataError(f"Yahoo 沒有 {symbol} 的資券餘額資料 (0 筆，可能是不能融資融券的股票)")
        try:
            items = sorted(items, key=lambda r: _api_date(r["date"]), reverse=True)[:MARGIN_ROWS]
            dates = [_api_date(r["date"]) for r in items]  # 每週的週一 (與網頁相同)
            margin_rows, lending_rows = [], []
            for r in items:
                short_change = _fmt_change(r, "shortSellVolK", "shortBuyVolK", "shortRepayVolK")
                margin_rows.append([
                    None, "", _fmt_change(r, "financingBuyVolK", "financingSellVolK", "financingPaybackVolK"),
                    _fmt_int(r.get("financingTotalVolK")), _fmt_ratio(r.get("financingUsageRatio")),
                    "", short_change, _fmt_int(r.get("shortTotalVolK")), _fmt_ratio(r.get("shortUsageRatio")),
                    _fmt_percent(r.get("shortFinancingPercent")), _fmt_int(r.get("dayTradingVolK")),
                ])
                lending_rows.append([
                    None, "", short_change, _fmt_int(r.get("shortTotalVolK")), _fmt_int(r.get("shortLimitVolK")),
                    "", _fmt_change(r, "lendingSellVolK", "lendingRepayVolK"), _fmt_int(r.get("lendingTotalVolK")),
                ])
        except (AttributeError, KeyError, TypeError, ValueError) as e:
            raise _api_format_error(symbol, label, e) from None
        _api_check_weeks(dates, symbol, label)
        return (margin_rows, lending_rows), dates

    def read_latest_day():
        items = credits("day", API_DAILY_ROWS)
        try:
            return max((_api_date(r["date"]) for r in items), default=None)
        except (AttributeError, KeyError, TypeError, ValueError) as e:
            raise _api_format_error(symbol, label, e) from None

    (margin_rows, lending_rows), dates = _api_read_fresh(symbol, label, read_week, read_latest_day)
    return (
        _to_week_df(margin_rows, dates, MARGIN_COLUMNS, symbol),
        _to_week_df(lending_rows, dates, LENDING_COLUMNS, symbol),
    )


def _api_major_holders(symbol, timeout_s):
    label = "大戶籌碼"
    data = _api_get(
        f"StockServices.holderStats;includedFields=quoteStats;limit=52;sortBy=-date;symbol={symbol}",
        f"{symbol} 大戶籌碼", timeout_s,
    )
    cutoff = date.today() - timedelta(days=MAJOR_HOLDERS_DAYS)  # 網頁只顯示最近一年
    try:
        items = [r for r in data["data"]["list"] if _api_date(r.get("startDate") or r["endDate"]) >= cutoff]
        items.sort(key=lambda r: _api_date(r["endDate"]), reverse=True)
        dates = [_api_date(r["endDate"]) for r in items]
        rows = [
            [None, _fmt_percent(r.get("foreignHoldPercent")), _fmt_percent(r.get("mainHoldPercent")),
             _fmt_percent(r.get("dirSupHoldPercent")), _fmt_price_text((r.get("quoteStats") or {}).get("closePrice"))]
            for r in items
        ]
    except (AttributeError, KeyError, TypeError, ValueError) as e:
        raise _api_format_error(symbol, label, e) from None
    _api_check_weeks(dates, symbol, label)
    return _to_week_df(rows, dates, MAJOR_HOLDERS_COLUMNS, symbol)


def _api_chart_bars(symbol, period, timeout_s):
    """K 線資料 → [(日期, [開, 高, 低, 收]), ...]，日期由舊到新；period："w" 週、"d" 日"""
    symbols = url_quote(json.dumps([symbol], separators=(",", ":")))
    data = _api_get(
        f"FinanceChartService.ApacLibraCharts;period={period};symbols={symbols}",
        f"{symbol} {'週' if period == 'w' else '日'}K線", timeout_s,
    )
    taipei = timezone(timedelta(hours=8))  # K 棒的時間戳記是台北時間當天 0 點
    try:
        chart = data["data"][0]["chart"] if data.get("data") else {}
        stamps = chart.get("timestamp") or []
        quote = chart["indicators"]["quote"][0] if stamps else {}
        bars = []
        for i, ts in enumerate(stamps):
            ohlc = [quote[k][i] for k in ("open", "high", "low", "close")]
            if None not in ohlc:  # 沒有成交的週沒有 K 棒
                bars.append((datetime.fromtimestamp(ts, taipei).date(), ohlc))
    except (AttributeError, KeyError, TypeError, ValueError, IndexError) as e:
        raise _api_format_error(symbol, "K線", e) from None
    bars.sort(key=lambda b: b[0])
    return bars


def _api_stock_price(symbol, timeout_s):
    label = "週K線"

    def read_week():
        bars = _api_chart_bars(symbol, "w", timeout_s)[::-1][:PRICE_ROWS]  # 由新到舊，最近 120 週
        dates = [d for d, _ in bars]
        _api_check_weeks(dates, symbol, label)
        return [[None] + [_fmt_price_number(v) for v in ohlc] for _, ohlc in bars], dates

    def read_latest_day():
        bars = _api_chart_bars(symbol, "d", timeout_s)
        return bars[-1][0] if bars else None

    rows, dates = _api_read_fresh(symbol, label, read_week, read_latest_day)
    return _to_week_df(rows, dates, PRICE_COLUMNS, symbol)


# --- 對外的爬蟲函式 (名稱、參數與以前相同) ---
def fetch_institutional_trading_week(abbr, headless=True, timeout_s=30):
    """取得「法人逐日買賣超」→「週」的資料，回傳 DataFrame (JSON 介面)
    headless=False 時改用瀏覽器版 (可以看到操作過程)"""
    if not headless:
        return fetch_institutional_trading_week_browser(abbr, headless=False, timeout_s=timeout_s)
    return _api_institutional(_api_symbol(abbr), timeout_s)


def fetch_margin_week(abbr, headless=True, timeout_s=30):
    """取得「資券餘額逐日增減」→「週」的兩份資料，回傳 (資券餘額 DataFrame, 借券賣出餘額 DataFrame) (JSON 介面)
    headless=False 時改用瀏覽器版"""
    if not headless:
        return fetch_margin_week_browser(abbr, headless=False, timeout_s=timeout_s)
    return _api_margin(_api_symbol(abbr), timeout_s)


def fetch_major_holders_week(abbr, headless=True, timeout_s=30):
    """取得「大戶籌碼」的週資料 (最近一年)，回傳 DataFrame (JSON 介面)
    headless=False 時改用瀏覽器版"""
    if not headless:
        return fetch_major_holders_week_browser(abbr, headless=False, timeout_s=timeout_s)
    return _api_major_holders(_api_symbol(abbr), timeout_s)


def fetch_stock_price_week(abbr, headless=True, timeout_s=30):
    """取得「週」K 線的 日期/開/高/低/收 (最近 120 週，日期新到舊)，回傳 DataFrame (JSON 介面)
    headless=False 時改用瀏覽器版"""
    if not headless:
        return fetch_stock_price_week_browser(abbr, headless=False, timeout_s=timeout_s)
    return _api_stock_price(_api_symbol(abbr), timeout_s)

In [6]:
# ===== 全部週資料（5 份 csv）：一次取得 =====

# 每份週資料的檔名標籤: 欄位 (檔名格式為 ABBR_{檔名標籤}_檔案建立年月日.csv)
WEEK_FILE_COLUMNS = {
    "Institutional_Trading_Week_8cols": INSTITUTIONAL_COLUMNS,
    "Margin_Week_11cols": MARGIN_COLUMNS,
    "Securities_Lending_and_Borrowing_Week_8cols": LENDING_COLUMNS,
    "Major_Holders_Week_5cols": MAJOR_HOLDERS_COLUMNS,
    PRICE_FILE_TAG: PRICE_COLUMNS,
}


def fetch_all_week_data(abbr, headless=True, timeout_s=30, source="api"):
    """依序取得法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價 5 份週資料，回傳 {檔名標籤: DataFrame}
    (任何一份失敗就直接報錯，不會只回傳部分資料)
    source="api"：JSON 介面 (預設，約 2 秒)；source="browser"：瀏覽器爬蟲 (約 1 分鐘，Yahoo 介面失效時的備案)
    headless=False (想看瀏覽器操作過程) 時一律使用瀏覽器版"""
    if source == "browser" or not headless:
        fetchers = (fetch_institutional_trading_week_browser, fetch_margin_week_browser,
                    fetch_major_holders_week_browser, fetch_stock_price_week_browser)
    elif source == "api":
        fetchers = (fetch_institutional_trading_week, fetch_margin_week, fetch_major_holders_week, fetch_stock_price_week)
    else:
        raise ValueError(f'source 只能是 "api" 或 "browser"，收到 {source!r}')
    fetch_inst, fetch_margin, fetch_major, fetch_price = fetchers
    df_inst = fetch_inst(abbr, headless=headless, timeout_s=timeout_s)
    df_margin, df_lending = fetch_margin(abbr, headless=headless, timeout_s=timeout_s)
    df_major = fetch_major(abbr, headless=headless, timeout_s=timeout_s)
    df_price = fetch_price(abbr, headless=headless, timeout_s=timeout_s)
    return {
        "Institutional_Trading_Week_8cols": df_inst,
        "Margin_Week_11cols": df_margin,
        "Securities_Lending_and_Borrowing_Week_8cols": df_lending,
        "Major_Holders_Week_5cols": df_major,
        PRICE_FILE_TAG: df_price,
    }

### 輸入公司代碼，開始蒐集資料

In [8]:
# ===== 執行：輸入代碼 → 取得「週」法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價 → 存檔 =====
ABBR = ask_stock_code()

# 預設用 JSON 介面 (約 2 秒)；Yahoo 介面失效時可改用 fetch_all_week_data(ABBR, source="browser") (約 1 分鐘)，
# 想看瀏覽器操作過程可加 headless=False
week_data = fetch_all_week_data(ABBR)

# 5 份資料都抓成功後才清掉舊檔，讓 DataCollection/ABBR 資料夾內只保留最新的 5 份 csv
for old in clear_week_csv(ABBR, week_data):
    print(f"已刪除舊檔：{old.resolve()}")

for file_tag, df in week_data.items():
    csv_path = save_week_csv(df, ABBR, file_tag)
    print(f"{df.attrs['symbol']} {file_tag} 共 {len(df)} 筆，已存檔：{csv_path.resolve()}")
    display(df.head(3))

請輸入台股代碼(4位數字)：5274
5274.TWO Institutional_Trading_Week_8cols 共 52 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\DataCollection\5274\5274_Institutional_Trading_Week_8cols_20261127.csv


,日期,外資(張),投信(張),自營商(張),合計(張),外資籌碼,漲跌幅(%),成交量
0,2026/11/26,35,-80,0,-45,51.74%,6.65%,"1,911"
1,2026/11/20,265,-339,5,-69,51.67%,-9.97%,"2,268"
2,2026/11/13,-74,-84,-2,-159,51.00%,0.38%,"3,057"


5274.TWO Margin_Week_11cols 共 30 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\DataCollection\5274\5274_Margin_Week_11cols_20261127.csv


,日期,融資,增減,餘額,使用率％,融券,增減,餘額,使用率％,券資比,資券互抵
0,2026/11/23,,-5,140,1.38%,,0,2,0.02%,1.43%,0
1,2026/11/16,,-29,145,1.73%,,2,2,0.02%,1.38%,0
2,2026/11/9,,-6,174,1.63%,,0,0,0.02%,0.00%,0


5274.TWO Securities_Lending_and_Borrowing_Week_8cols 共 30 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\DataCollection\5274\5274_Securities_Lending_and_Borrowing_Week_8cols_20261127.csv


,日期,融券,增減,餘額,限額,借券賣出,增減,餘額
0,2026/11/23,,0,2,-,,3,211
1,2026/11/16,,2,2,-,,-68,208
2,2026/11/9,,0,0,-,,52,276


5274.TWO Major_Holders_Week_5cols 共 50 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\DataCollection\5274\5274_Major_Holders_Week_5cols_20261127.csv


,年度/日期,外資籌碼,大戶籌碼,董監持股,股價
0,2026/11/20,51.67%,21.38%,20.39%,"7,860"
1,2026/11/13,51.00%,21.59%,20.39%,"12,000"
2,2026/11/6,51.15%,21.70%,20.39%,"11,375"


5274.TWO Stock_Price_Week 共 120 筆，已存檔：C:\Users\Michael\workplace\python\WeekTermInvestment\DataCollection\5274\5274_Stock_Price_Week_20261127.csv


,日期,開,高,低,收
0,2026/11/26,7605,9590,7295,9380
1,2026/11/20,11675,11875,7860,7860
2,2026/11/13,11155,13375,10710,12000
